# 28 - Upskilling

This notebook implements the Chapter 4 v1 recommender workflow end-to-end to validate the logic before hardening it in `src/`. It starts from a user profile (free-text skills + constraints like state, title family, sector, salary target), runs the hybrid recommender to produce a **frozen scored universe** of jobs, and then generates two recommendation buckets (**best_now** and **stretch**) ranked by the Chapter 4 score. It then runs the explanation layer to attach interpretable fields (bucket/rank rationale + missing skill families) and extracts missing skill-family tokens from job descriptions to build upskilling candidates. Finally, it iterates through counterfactual “add one missing family” scenarios, recomputes scores on the same frozen universe, stacks all scenarios into a long table, and computes per-scenario deltas (score, suitability, competitiveness, bucket movement) to rank upskilling actions via an ROI-like impact score.


## Libraries

In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
#===
import sys
from pathlib import Path

## Path

In [2]:
project_root = Path().resolve().parent.parent
sys.path.append(str(project_root))
project_root

PosixPath('/Users/alejandrofp/Desktop/Projects/03_Flagship_Portfolio/job-intelligence-engine')

In [3]:
from src.job_intel.features.job_recommender import job_recommender
from src.job_intel.features.job_explanations import build_job_explanations
from src.job_intel.features.skill_extractor import explain_matches

### Compute the candidate's current jobs and indices

In [4]:
skill_text= "python, sql, bayesian, communication, problem-solving, research, publication, causal inference, statistical modelling," \
"r, ecology, visualisaion, ggpplot, seaborn, numpy, pandas, git, github, microsoft office, phd, neural networks, excell, teamwork, team member, cloud, aws" \
"pca, recommender systems, shiny app, shiny, technical writting, scientific research"
current_state= ("ALL")
job_title_family = "data_scientist"
job_title_rich= None
target_sectors = None
salary_target = 200000
explain_skills = False


baseline_recommnedations = job_recommender(skill_text=skill_text,
                             current_state=current_state,
                             job_title_family=job_title_family,
                             job_title_rich=job_title_rich,
                             target_sectors=target_sectors,
                             salary_target=salary_target,
                             explain_skills=explain_skills,
                             verbose=False)

override_df = baseline_recommnedations["tables"]["scored_universe"][["job_id"]].copy()

explanations = build_job_explanations(rec = baseline_recommnedations)

### Baseline indices

In [5]:
# Baseline table containing both top best and top stretch jobs

jobs = baseline_recommnedations['tables']['scored_universe'][['job_id', 'Job Description', 'skill_match_score', 'skill_match_norm', 'salary_score','suitability','expected_missing', 'expected_missing_norm', 'salary_pct', 'competitiveness_index', 'pred_sal', 'bucket', 'score']].copy()
jobs['upskill_scenario'] = 'baseline' # define a column with the version (baseline = user input)

# Retrieve the explanation stretch jobs where the missing families are. Missing families are stored in a list within the cell

stretch_jobs = explanations['tables']['scored_universe_explained'][['job_id','bucket','missing_families','n_missing_families']]
stretch_jobs = stretch_jobs[stretch_jobs['bucket'] == 'stretch']
stretch_jobs = stretch_jobs.merge(jobs[['job_id', 'Job Description']], how = 'left', on = 'job_id')

# Append missing tokens to the missing families
stretch_jobs['matches_dict'] = stretch_jobs['Job Description'].fillna("").apply(lambda x: explain_matches(x))
    # Set of missing families
stretch_jobs['set_missing_families'] = stretch_jobs['missing_families'].apply(lambda x: set(x))

stretch_jobs["missing_matches_dict"] = [
    {fam: toks for fam, toks in md.items() if fam in miss}
    for md, miss in zip(stretch_jobs["matches_dict"], stretch_jobs["set_missing_families"])
]


missing_dict = {}

for i in range(len(stretch_jobs)):
    new_dict = stretch_jobs["missing_matches_dict"].iloc[i]  # dict: family -> list[tokens]

    for key, val in new_dict.items():
        if key not in missing_dict:
            missing_dict[key] = list(val)          # copy list
        else:
            missing_dict[key].extend(val)          # append tokens

set(missing_dict.keys()) == set().union(*stretch_jobs["set_missing_families"])

True

In [6]:
jobs['score']

0       0.450355
1       0.543694
2       0.623260
3       0.259642
4       0.448027
          ...   
1297    0.369326
1298    0.304873
1299    0.514214
1300    0.331048
1301    0.510222
Name: score, Length: 1302, dtype: float64

## Loop over missing skills

In [7]:
upskill_rec_df = jobs

for key in missing_dict.keys():

    tok = list(dict.fromkeys(missing_dict[key]))[:3]   # dedup, keep first 3
    upskill_text = skill_text + " " + " ".join(tok)
    
    recommnedations_upskill = job_recommender(skill_text=upskill_text,
                                                current_state=current_state,
                                                job_title_family=job_title_family,
                                                job_title_rich=job_title_rich,
                                                target_sectors=target_sectors,
                                                salary_target=salary_target,
                                                explain_skills=explain_skills,
                                                return_top_n_jobs = None,
                                                verbose=False,
                                                candidate_override_df = override_df)
    
    jobs_upskill = recommnedations_upskill['tables']['scored_universe'][['job_id', 'Job Description', 'skill_match_score', 'skill_match_norm', 'salary_score','suitability','expected_missing', 'expected_missing_norm', 'salary_pct', 'competitiveness_index', 'pred_sal', 'bucket', 'score']].copy()
    jobs_upskill['upskill_scenario'] = 'upskill_'+ key # define a column with the version (baseline = user input)
    upskill_rec_df = pd.concat([upskill_rec_df, jobs_upskill], axis = 0)

In [8]:
sum(upskill_rec_df['upskill_scenario'].value_counts() == len(jobs)) / len(missing_dict.keys()) == 1

False

In [9]:
baseline = jobs[['job_id',
                 'skill_match_norm',
                 'suitability',
                 'expected_missing_norm', 
                 'competitiveness_index',
                 'bucket', 
                 'score']].rename(columns = {
                                                 'skill_match_norm': 'baseline_skill_match_norm',
                                                 'suitability':'baseline_suitability', 
                                                 'expected_missing_norm':'baseline_expected_missing_norm', 
                                                 'competitiveness_index':'baseline_competitiveness_index',
                                                 'bucket':'baseline_bucket', 
                                                 'score':'baseline_score'})
upskill_rec_df = upskill_rec_df[upskill_rec_df['upskill_scenario'] != 'baseline']

upskill_df = upskill_rec_df.merge(baseline, how = 'left', on = 'job_id')


In [10]:
upskill_df

,job_id,Job Description,skill_match_score,skill_match_norm,salary_score,suitability,expected_missing,expected_missing_norm,salary_pct,competitiveness_index,pred_sal,bucket,score,upskill_scenario,baseline_skill_match_norm,baseline_suitability,baseline_expected_missing_norm,baseline_competitiveness_index,baseline_bucket,baseline_score
0,0,"ABOUT HOPPER\n\nAt Hopper, we’re on a mission ...",0.239669,0.619835,0.7300,0.652884,0.092144,0.003413,0.771121,0.387267,125617.671875,best_now,0.459251,upskill_core_programming__intermediate,0.607461,0.644223,0.004349,0.387735,best_now,0.450355
1,1,"At Noom, we use scientifically proven methods ...",0.472828,0.736414,0.7300,0.734490,0.210465,0.007795,0.771121,0.389458,129626.640625,best_now,0.539761,upskill_core_programming__intermediate,0.742080,0.738456,0.007925,0.389523,best_now,0.543694
2,10,Company Description:\n\nQuartet is a pioneerin...,0.704263,0.852132,0.7300,0.815492,0.505553,0.018724,0.771121,0.394923,129081.789062,best_now,0.618031,upskill_core_programming__intermediate,0.859674,0.820772,0.018925,0.395023,best_now,0.623260
3,100,About Point72\nPoint72 Asset Management is a g...,-0.447364,0.276318,0.4950,0.341923,0.148715,0.005508,0.340630,0.173069,111901.976562,best_now,0.255388,upskill_core_programming__intermediate,0.282410,0.346187,0.005551,0.173091,best_now,0.259642
4,1003,Role : Informatica Data Modeler\nLocation: Chi...,0.102283,0.551142,0.4125,0.509549,0.270908,0.010034,0.232719,0.121376,94682.460938,best_now,0.448861,upskill_core_programming__intermediate,0.549955,0.508718,0.010045,0.121382,best_now,0.448027
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15619,96,Search by Keyword\n\nSearch by Location\n\nCle...,-0.107978,0.446011,0.4950,0.460708,0.653774,0.024214,0.340630,0.182422,127940.226562,best_now,0.369497,upskill_core_programming__advanced,0.445767,0.460537,0.024214,0.182422,best_now,0.369326
15620,960,Uptake is a Chicago-based predictive analytics...,-0.336022,0.331989,0.2825,0.317142,0.131261,0.004862,0.045699,0.025280,94548.515625,best_now,0.304502,upskill_core_programming__advanced,0.332519,0.317513,0.004862,0.025280,best_now,0.304873
15621,97,Search by Keyword\n\nSearch by Location\n\nCle...,0.297785,0.648892,0.4950,0.602725,0.397405,0.014719,0.340630,0.177674,127940.226562,best_now,0.513888,upskill_core_programming__advanced,0.649359,0.603052,0.014719,0.177674,best_now,0.514214
15622,977,About JLL Technologies\n\nJLL is a leading pro...,-0.241790,0.379105,0.2425,0.338123,0.302514,0.011204,0.019969,0.015587,96481.648438,best_now,0.330330,upskill_core_programming__advanced,0.380130,0.338841,0.011204,0.015587,best_now,0.331048


In [11]:
upskill_df['delta_skill_match_norm'] = (upskill_df['skill_match_norm'] - upskill_df['baseline_skill_match_norm']) * 100
upskill_df['delta_suitability'] = (upskill_df['suitability'] - upskill_df['baseline_suitability'])*100
upskill_df['delta_expected_missing_norm'] = (upskill_df['expected_missing_norm'] - upskill_df['baseline_expected_missing_norm'])*100
upskill_df['delta_competitiveness_index'] = (upskill_df['competitiveness_index'] - upskill_df['baseline_competitiveness_index'])*100
upskill_df['delta_score'] = (upskill_df['score'] - upskill_df['baseline_score'])*100

upskill_df["bucket_movement"] = upskill_df.apply(
    lambda row: "unchanged"
    if row["bucket"] == row["baseline_bucket"]
    else "promoted"
    if (row["bucket"] == "best_now" and row["baseline_bucket"] == "stretch")
    else "demoted",
    axis=1,
)

baseline_buckets = baseline.groupby('baseline_bucket')['job_id'].count().reset_index().rename(columns = {'baseline_bucket': 'bucket',
                                                                                                         'job_id':'n'})

In [12]:
scenario_bucket = upskill_df.groupby(['upskill_scenario', 'bucket']).count().reset_index()[['upskill_scenario', 'bucket', 'bucket_movement']]

scenario_bucket = scenario_bucket.merge(baseline_buckets, how = 'left', on = 'bucket')

scenario_bucket['delta_movement'] = scenario_bucket['bucket_movement'] - scenario_bucket['n']

scenario_bucket_pivot = scenario_bucket[['upskill_scenario', 'bucket', 'delta_movement']].pivot(index='upskill_scenario', columns='bucket', values='delta_movement').rename(columns = {'best_now': 'delta_best', 'stretch':'delta_stretch'})

In [13]:
upskill_summary = upskill_df.groupby('upskill_scenario').agg(delta_skill_match_norm_mean = ('delta_skill_match_norm', 'mean'),
                                           delta_suitability_mean = ('delta_suitability', 'mean'),
                                           delta_expected_missing_norm_mean = ('delta_expected_missing_norm', 'mean'),
                                           delta_competitiveness_index_mean = ('delta_competitiveness_index', 'mean'),
                                           delta_score_mean = ('delta_score', 'mean'),
                                           )

In [14]:
upskill_summary = upskill_summary.merge(scenario_bucket_pivot, how='left', on = 'upskill_scenario')

In [15]:
upskill_recommendation = upskill_summary.sort_values(by = ['delta_score_mean', 'delta_competitiveness_index_mean', 'delta_stretch'], ascending=[False,True,True]).head(3)

recommended_skill_families = list(upskill_recommendation.index)


In [16]:
recommendation_dict = {}

for scenario in recommended_skill_families:
    family = scenario.replace("upskill_", "", 1)   # map scenario -> raw family key

    tokens = missing_dict.get(family, [])
    seen = set()
    deduped = []
    for tok in tokens:
        t = str(tok).strip()
        if not t:
            continue
        k = t.lower()
        if k in seen:
            continue
        seen.add(k)
        deduped.append(t)

    recommendation_dict[family] = deduped[:5]  # choose how many examples you want

recommendation_dict


{'core_programming__intermediate': ['scala', 'julia'],
 'analytics_stats__intermediate': ['statistical analysis',
  'marketing analytics',
  'customer analytics'],
 'data_engineering_pipelines__advanced': ['mapreduce',
  'scalable',
  'scalability']}

In [17]:
upskill_df

,job_id,Job Description,skill_match_score,skill_match_norm,salary_score,suitability,expected_missing,expected_missing_norm,salary_pct,competitiveness_index,...,baseline_expected_missing_norm,baseline_competitiveness_index,baseline_bucket,baseline_score,delta_skill_match_norm,delta_suitability,delta_expected_missing_norm,delta_competitiveness_index,delta_score,bucket_movement
0,0,"ABOUT HOPPER\n\nAt Hopper, we’re on a mission ...",0.239669,0.619835,0.7300,0.652884,0.092144,0.003413,0.771121,0.387267,...,0.004349,0.387735,best_now,0.450355,1.237365,0.866156,-9.363991e-02,-4.681996e-02,0.889566,unchanged
1,1,"At Noom, we use scientifically proven methods ...",0.472828,0.736414,0.7300,0.734490,0.210465,0.007795,0.771121,0.389458,...,0.007925,0.389523,best_now,0.543694,-0.566583,-0.396608,-1.295926e-02,-6.479628e-03,-0.393368,unchanged
2,10,Company Description:\n\nQuartet is a pioneerin...,0.704263,0.852132,0.7300,0.815492,0.505553,0.018724,0.771121,0.394923,...,0.018925,0.395023,best_now,0.623260,-0.754207,-0.527945,-2.009030e-02,-1.004515e-02,-0.522923,unchanged
3,100,About Point72\nPoint72 Asset Management is a g...,-0.447364,0.276318,0.4950,0.341923,0.148715,0.005508,0.340630,0.173069,...,0.005551,0.173091,best_now,0.259642,-0.609194,-0.426436,-4.345505e-03,-2.172753e-03,-0.425350,unchanged
4,1003,Role : Informatica Data Modeler\nLocation: Chi...,0.102283,0.551142,0.4125,0.509549,0.270908,0.010034,0.232719,0.121376,...,0.010045,0.121382,best_now,0.448027,0.118673,0.083071,-1.127500e-03,-5.637502e-04,0.083353,unchanged
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15619,96,Search by Keyword\n\nSearch by Location\n\nCle...,-0.107978,0.446011,0.4950,0.460708,0.653774,0.024214,0.340630,0.182422,...,0.024214,0.182422,best_now,0.369326,0.024411,0.017088,-1.085865e-07,-5.429325e-08,0.017088,unchanged
15620,960,Uptake is a Chicago-based predictive analytics...,-0.336022,0.331989,0.2825,0.317142,0.131261,0.004862,0.045699,0.025280,...,0.004862,0.025280,best_now,0.304873,-0.052991,-0.037094,-3.500991e-07,-1.750495e-07,-0.037094,unchanged
15621,97,Search by Keyword\n\nSearch by Location\n\nCle...,0.297785,0.648892,0.4950,0.602725,0.397405,0.014719,0.340630,0.177674,...,0.014719,0.177674,best_now,0.514214,-0.046697,-0.032688,-2.129169e-07,-1.064584e-07,-0.032688,unchanged
15622,977,About JLL Technologies\n\nJLL is a leading pro...,-0.241790,0.379105,0.2425,0.338123,0.302514,0.011204,0.019969,0.015587,...,0.011204,0.015587,best_now,0.331048,-0.102507,-0.071755,-2.972383e-06,-1.486192e-06,-0.071754,unchanged


## Report

In [18]:
upskill_recommendation

,delta_skill_match_norm_mean,delta_suitability_mean,delta_expected_missing_norm_mean,delta_competitiveness_index_mean,delta_score_mean,delta_best,delta_stretch
upskill_scenario,,,,,,,
upskill_core_programming__intermediate,0.614579,0.430206,-0.145193,-0.072597,0.466504,1,-1
upskill_analytics_stats__intermediate,0.437983,0.306588,-0.146136,-0.073068,0.343122,3,-3
upskill_data_engineering_pipelines__advanced,0.328088,0.229661,-0.102245,-0.051123,0.255223,3,-3


In [19]:
print('Upskilling report')
print('-----------------')
print('\n')
print('Overview:\n')
print('Based on the user inputs and constraints. The Job Recommender System retrieved the top recommended jobs and ' \
'placed them in two buckets. \nThe "best-now" bucket contained the most suitable jobs based on the user current profile.' \
'\nThe "stretch" bucket contained jobs currently out of reach but close to the high-suitability threshold. \nUsing the "stretch"' \
'jobs, the Engine retrieved the missing skills that could position the user closer to be competitive for those stretch options.\n' \
'Then, the Engine iterated over the missing skills recomputing the suitability and competitiveness scores given that the user had that skill. ' \
'\nThis provides the upskilling recommendation ground. Missing skill families were then ranked based on how much they increase the user position within the ' \
'\nmarket universe constrained by the user options (e.g., State, Sector, etc), providing the higher ROI skills that the Engine would recommend for the most optimal upskilling.')
print('\nGiven this approach, the top 3 skill families for upskilling are:\n')
for key, val in recommendation_dict.items():
    print(f"* {key} : {val}")
print('\nUpskilling score per job family:\n')
print(f'- Upskilling {recommended_skill_families[0].replace("upskill_", "", 1)} increases the user score by {round(upskill_recommendation.iloc[0]['delta_score_mean'],2)} percentage points.')
print(f'- Upskilling {recommended_skill_families[1].replace("upskill_", "", 1)} increases the user score by {round(upskill_recommendation.iloc[1]['delta_score_mean'],2)} percentage points.')
print(f'- Upskilling {recommended_skill_families[2].replace("upskill_", "", 1)} increases the user score by {round(upskill_recommendation.iloc[2]['delta_score_mean'],2)} percentage points.')

Upskilling report
-----------------


Overview:

Based on the user inputs and constraints. The Job Recommender System retrieved the top recommended jobs and placed them in two buckets. 
The "best-now" bucket contained the most suitable jobs based on the user current profile.
The "stretch" bucket contained jobs currently out of reach but close to the high-suitability threshold. 
Using the "stretch"jobs, the Engine retrieved the missing skills that could position the user closer to be competitive for those stretch options.
Then, the Engine iterated over the missing skills recomputing the suitability and competitiveness scores given that the user had that skill. 
This provides the upskilling recommendation ground. Missing skill families were then ranked based on how much they increase the user position within the 
market universe constrained by the user options (e.g., State, Sector, etc), providing the higher ROI skills that the Engine would recommend for the most optimal upskilling.

Giv

## Test src

In [20]:
from src.job_intel.features.upskilling_recommender import upskill_recommender

In [21]:
skill_text= "python, sql, bayesian, communication, problem-solving, research, publication, causal inference, statistical modelling," \
"r, ecology, visualisaion, ggpplot, seaborn, numpy, pandas, git, github, microsoft office, phd, neural networks, excell, teamwork, team member, cloud, aws" \
"pca, recommender systems, shiny app, shiny, technical writting, scientific research"
current_state= ("ALL")
job_title_family = "data_scientist"
job_title_rich= None
target_sectors = None
salary_target = 200000
explain_skills = False

out = upskill_recommender(skill_text=skill_text,
                             current_state=current_state,
                             job_title_family=job_title_family,
                             job_title_rich=job_title_rich,
                             target_sectors=target_sectors,
                             salary_target=salary_target,
                             explain_skills=explain_skills)

Upskilling report
-----------------

Ranking logic:
- Universe is frozen by job_id (candidate_override_df), so deltas are comparable.
- Missing families come from explained stretch jobs (missing_families).
- Each scenario injects representative tokens for one family into skill_text.
- Scenarios are skipped if the injected tokens do not change the extracted user skill_vector.
- Deltas are percentage points (bounded 0–1 metrics × 100).
- Ranking rewards promotions + score gains (esp. baseline-stretch), penalises demotions + worst-tail harms.
- Guardrail: demotion_rate <= 0.0.

Top 3 recommended skill families (with example tokens):
* cloud__advanced: ['gpu', 'parallel computing']
* analytics_stats__intermediate: ['statistical analysis', 'marketing analytics', 'customer analytics']
* core_programming__advanced: ['multithreading']

Top scenarios summary:
                                       upskill_impact_score  promotion_rate  \
upskill_scenario                                          

In [22]:
out["scenario_meta"].sort_values(["status", "family"]).head(50)

,family,status,reason,tokens
2,analytics_stats__intermediate,kept,None,"[statistical analysis, marketing analytics, cu..."
4,cloud__advanced,kept,None,"[gpu, parallel computing]"
3,cloud__intermediate,kept,None,"[devops, distributed computing]"
11,core_programming__advanced,kept,None,[multithreading]
0,core_programming__intermediate,kept,None,"[scala, julia]"
1,data_engineering_pipelines__advanced,kept,None,"[mapreduce, scalable, scalability]"
7,data_engineering_pipelines__intermediate,kept,None,"[spark, data ingestion, big data]"
8,db_storage__advanced,kept,None,[databricks]
10,db_storage__intermediate,kept,None,"[mongodb, mysql, oracle]"
6,domain_specific__none,kept,None,"[electronics, electrical engineering, saas]"


## Test career simulator

In [28]:
from job_intel.v2_updates.features.career_simulator import (
    career_simulation,
    SimulationScenario,
    SimulationConfig,
)

ModuleNotFoundError: No module named 'job_intel'

In [29]:
# --- Notebook setup (required for src/ imports) ---
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()  # adjust if your notebook isn't run from repo root
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# --- Inputs (fix: make this ONE string; your "\" concatenation was joining "aws" + "pca" into "awspca") ---
skill_text = (
    "python, sql, bayesian, communication, problem-solving, research, publication, causal inference, "
    "statistical modelling, r, ecology, visualisation, ggplot, seaborn, numpy, pandas, git, github, "
    "microsoft office, phd, neural networks, excel, teamwork, cloud, aws, pca, recommender systems, "
    "shiny app, shiny, technical writing, scientific research"
)

current_state = "ALL"
job_title_family = "data_scientist"
job_title_rich = None
target_sectors = None
salary_target = 200000
explain_skills = False

# --- Scenarios (fix: must be list[SimulationScenario], not list[str]) ---
scenarios = [
    SimulationScenario(
        name="parallelism_tooling",
        added_tokens=["parallel computing", "multithreading", "pycharm"],
        max_tokens=3,
        demotion_tol=0.0,
    )
]

config = SimulationConfig(
    top_n_unlocked_jobs=20,
    require_frozen_universe=True,
)

# --- Run ---
simulation = career_simulation(
    skill_text=skill_text,
    current_state=current_state,
    job_title_family=job_title_family,
    job_title_rich=job_title_rich,
    target_sectors=target_sectors,
    salary_target=salary_target,
    explain_skills=explain_skills,
    scenarios=scenarios,
    config=config,
)

# --- Inspect ---
simulation["scenario_meta"]
# If this shows "skipped" with reason like "skill_vector_no_effect", your tokens are likely out-of-vocabulary in v1.

simulation["scenario_summary"].sort_values(
    by=["passes_guardrail", "promotion_rate"],
    ascending=[False, False],
)

simulation["top_unlocked_jobs"].head(20)


,job_id,skill_match_norm,suitability,expected_missing_norm,competitiveness_index,bucket,score,scenario,baseline_skill_match_norm,baseline_suitability,...,baseline_competitiveness_index,baseline_bucket,baseline_score,delta_skill_match_norm,delta_suitability,delta_expected_missing_norm,delta_competitiveness_index,delta_score,bucket_movement,state
0,200,0.564481,0.594637,0.000999,0.337289,best_now,0.425992,parallelism_tooling,0.561365,0.592455,...,0.546323,stretch,0.319294,0.311679,0.218175,-41.806814,-20.903407,10.669879,promoted,NY
1,1205,0.852928,0.863300,0.034908,0.490764,best_now,0.617918,parallelism_tooling,0.844264,0.857235,...,0.506442,stretch,0.604014,0.866428,0.606500,-3.135475,-1.567738,1.390368,promoted,TX
2,185,0.630904,0.741633,0.009356,0.494693,best_now,0.494286,parallelism_tooling,0.622816,0.735971,...,0.510498,stretch,0.480722,0.808806,0.566164,-3.161057,-1.580528,1.356429,promoted,NY
3,193,0.814102,0.869871,0.009102,0.494566,best_now,0.622588,parallelism_tooling,0.814159,0.869911,...,0.510367,stretch,0.614728,-0.005714,-0.004000,-3.160055,-1.580027,0.786014,promoted,NY


# === End of Notebook ===